In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np




lf = np.array([ 0.0350, -0.2054,  0.2044,  0.0349,  0.0349, -0.1365])


sum(lf)

0.5/sum(lf)

sum(-15.29*lf)





In [ ]:
import numpy as np

def solve_for_a_and_b(X, Y):
    """
    Solve for scalars a and b in the equation Xa + b = Y using least squares.

    Parameters:
        X (array-like): 1D array of input values.
        Y (array-like): 1D array of target values.

    Returns:
        a (float): Coefficient scalar.
        b (float): Intercept scalar.
    """
    X = np.asarray(X)
    Y = np.asarray(Y)

    # Stack X with a column of ones to account for b (intercept)
    A = np.vstack([X, np.ones(len(X))]).T

    # Solve the least squares problem
    a, b = np.linalg.lstsq(A, Y, rcond=None)[0]

    return a, b

# Example usage
X = np.array([-0.0396,  0.2324, -0.2312, -0.0395, -0.0395])
Y = np.array([-0.0030, -0.1425,  0.0571, -0.0538, -0.0538]) # Expected a=2, b=1

a, b = solve_for_a_and_b(X, Y)
print(f"Solved a: {a}, b: {b}")












In [ ]:
X * -0.4255356180358302,3 + -0.049191576311481314

In [ ]:
import numpy as np

def solve_quadratic_fit(X, Y):
    """
    Solve for scalars a, b, c in the equation Y = aX^2 + bX + c using least squares.

    Parameters:
        X (array-like): 1D array of input values.
        Y (array-like): 1D array of target values.

    Returns:
        a (float): Coefficient of X^2.
        b (float): Coefficient of X.
        c (float): Constant term.
    """
    X = np.asarray(X)
    Y = np.asarray(Y)

    # Design matrix for quadratic: [X^2, X, 1]
    A = np.vstack([X**2, X, np.ones(len(X))]).T

    # Least squares solution
    a, b, c = np.linalg.lstsq(A, Y, rcond=None)[0]

    return a, b, c

# Example usage
X = np.array([-0.0396,  0.2324, -0.2312, -0.0395, -0.0395])
Y = np.array([-0.0030, -0.1425,  0.0571, -0.0538, -0.0538]) # Expected a=2, b=1

a, b, c = solve_quadratic_fit(X, Y)
print(f"Solved a: {a}, b: {b}, c: {c}")




In [ ]:
print( X**2 * a + X*b + c )

In [ ]:
import psi4
import numpy as np
import plotly.graph_objects as go

# Set up Psi4
psi4.set_memory('2 GB')
psi4.core.set_output_file('output.dat', False)

# Define molecule
mol = psi4.geometry("""
O
H 1 1.1
H 1 1.1 2 104
symmetry c1
""")

# Set options
psi4.set_options({'basis': '6-31g',
                  'scf_type': 'pk',
                  'mp2_type': 'conv',
                  'e_convergence': 1e-10,
                  'd_convergence': 1e-10})

# Run SCF calculation
scf_e, wfn = psi4.energy('SCF', return_wfn=True)

# Get initial MO coefficients
C = wfn.Ca()

# Boys localization
loc = psi4.core.Localizer.build('BOYS', wfn.basisset(), C)
loc.localize()
print("Boys Localization Convergedness: {}".format(loc.converged))

# Get localized orbitals
L = np.asarray(loc.L)
print("Orthogonality check L^T @ L:")
print(L.T @ L)

# # Update wavefunction with localized orbitals
# wfn.Ca().copy(psi4.core.Matrix.from_array(L))

def make_grid(xmin, xmax, ymin, ymax, zmin, zmax, npts):
    """Create a 3D grid for orbital evaluation"""
    x = np.linspace(xmin, xmax, npts)
    y = np.linspace(ymin, ymax, npts)
    z = np.linspace(zmin, zmax, npts)
    X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
    grid = np.vstack([X.ravel(), Y.ravel(), Z.ravel()]).T
    return grid, (X, Y, Z)

# Create grid
grid, (X, Y, Z) = make_grid(-4, 4, -4, 4, -4, 4, 30)

# Get basis set information
ao_funcs = wfn.basisset()
nbf = ao_funcs.nbf()
npoints = grid.shape[0]

# Initialize array to store AO values at each grid point
ao_values = np.zeros((npoints, nbf))

# Evaluate AO basis functions at each grid point
print("Evaluating AO basis functions on grid...")
for i, point in enumerate(grid):
    x, y, z = point
    # compute_phi returns a psi4.core.Vector, convert to numpy array
    phi_vals = np.array(ao_funcs.compute_phi(x, y, z))
    ao_values[i, :] = phi_vals
    
    # Progress indicator
    if i % 1000 == 0:
        print(f"Processed {i}/{npoints} points")

# Get MO coefficients (using localized orbitals)
C = L  # Use localized orbitals from Boys localization
C = np.asarray(wfn.Ca())


for i in range(nbf):
    # Choose which orbital to visualize (e.g., HOMO)
    homo_idx = wfn.nalpha() - 1  # HOMO index
    orb_coeff = C[:, i]  # Coefficients for chosen orbital

    print(f"Visualizing orbital {i} ")
    print(f"Orbital coefficients shape: {orb_coeff.shape}")
    print(f"AO values shape: {ao_values.shape}")

    # Evaluate orbital on grid: sum over basis functions
    orbital_vals = np.dot(ao_values, orb_coeff)  # shape (npoints,)

    # Reshape to 3D grid for visualization
    orbital_vals_3d = orbital_vals.reshape(X.shape)

    print(f"Orbital values range: {orbital_vals.min():.6f} to {orbital_vals.max():.6f}")
    print("Grid evaluation complete!")



    fig = go.Figure(data=go.Isosurface(
        x=X.flatten(), y=Y.flatten(), z=Z.flatten(),
        value=orbital_vals.flatten()**2,
        isomin=0.005,
        isomax=0.1,
        surface_count=3,
        colorscale='Viridis',
        caps=dict(x_show=False, y_show=False, z_show=False)
    ))
    fig.update_layout(title="HOMO Orbital Isosurface")
    fig.show()




# # Optional: Simple visualization with matplotlib
# try:
#     import matplotlib.pyplot as plt
    
#     # Plot a 2D slice through the molecule (z=0 plane)
#     z_center_idx = X.shape[2] // 2
#     slice_2d = orbital_vals_3d[:, :, z_center_idx]
    
#     plt.figure(figsize=(8, 6))
#     plt.contourf(X[:, :, z_center_idx], Y[:, :, z_center_idx], slice_2d, levels=20)
#     plt.colorbar(label='Orbital amplitude')
#     plt.xlabel('X (Bohr)')
#     plt.ylabel('Y (Bohr)')
#     plt.title(f'HOMO orbital (z=0 slice)')
#     plt.axis('equal')
#     plt.show()
    
# except ImportError:
#     print("Matplotlib not available for visualization")
#     print("Orbital data is stored in 'orbital_vals_3d' array")

In [ ]:
import VariationalLangFirsov_QEDCI as vlfqedci
import numpy as np


mol_str  = """
        H 0 0 0 
        He 0 0 1
        symmetry c1
        1 1
        """


# Set computation options
psi4_options = {'basis': 'sto-3g',
                  'scf_type': 'pk',
                  'e_convergence': 1e-12}

lambda_vector = np.array([0.0,0.0,0.05])



qedhf  = vlfqedci.QED_HF(mol_str=mol_str, psi_4_options_dict=psi4_options)
qedhf.qed_hf(lambda_vector=lambda_vector)


print("dipole matrix: " , qedhf.dipole_matrix)



wfn =  qedhf.wfn

# Get initial MO coefficients
C = wfn.Ca()

# Boys localization
loc = psi4.core.Localizer.build('BOYS', wfn.basisset(), C)
loc.localize()
print("Boys Localization Convergedness: {}".format(loc.converged))

# Get localized orbitals
L = np.asarray(loc.L)
print("Orthogonality check L^T @ L:")
print(L.T @ L)

# Update wavefunction with localized orbitals
# wfn.Ca().copy(psi4.core.Matrix.from_array(L))

#dipole basis C
#wfn.Ca().copy(psi4.core.Matrix.from_array(qedhf.C_dipole))


def make_grid(xmin, xmax, ymin, ymax, zmin, zmax, npts):
    """Create a 3D grid for orbital evaluation"""
    x = np.linspace(xmin, xmax, npts)
    y = np.linspace(ymin, ymax, npts)
    z = np.linspace(zmin, zmax, npts)
    X, Y, Z = np.meshgrid(x, y, z, indexing='ij')
    grid = np.vstack([X.ravel(), Y.ravel(), Z.ravel()]).T
    return grid, (X, Y, Z)

# Create grid
grid, (X, Y, Z) = make_grid(-4, 4, -4, 4, -4, 4, 30)

# Get basis set information
ao_funcs = wfn.basisset()
nbf = ao_funcs.nbf()
npoints = grid.shape[0]

# Initialize array to store AO values at each grid point
ao_values = np.zeros((npoints, nbf))

# Evaluate AO basis functions at each grid point
print("Evaluating AO basis functions on grid...")
for i, point in enumerate(grid):
    x, y, z = point
    # compute_phi returns a psi4.core.Vector, convert to numpy array
    phi_vals = np.array(ao_funcs.compute_phi(x, y, z))
    ao_values[i, :] = phi_vals
    
    # Progress indicator
    if i % 1000 == 0:
        print(f"Processed {i}/{npoints} points")

# Get MO coefficients (using localized orbitals)
C = L  # Use localized orbitals from Boys localization
C = np.asarray(wfn.Ca())


for i in range(nbf):
    # Choose which orbital to visualize (e.g., HOMO)
    homo_idx = wfn.nalpha() - 1  # HOMO index
    orb_coeff = C[:, i]  # Coefficients for chosen orbital

    print(f"Visualizing orbital {i} ")
    print(f"Orbital coefficients shape: {orb_coeff.shape}")
    print(f"AO values shape: {ao_values.shape}")

    # Evaluate orbital on grid: sum over basis functions
    orbital_vals = np.dot(ao_values, orb_coeff)  # shape (npoints,)

    # Reshape to 3D grid for visualization
    orbital_vals_3d = orbital_vals.reshape(X.shape)

    print(f"Orbital values range: {orbital_vals.min():.6f} to {orbital_vals.max():.6f}")
    print("Grid evaluation complete!")



    fig = go.Figure(data=go.Isosurface(
        x=X.flatten(), y=Y.flatten(), z=Z.flatten(),
        value=orbital_vals.flatten(),
        isomin=-0.1,
        isomax=0.1,
        surface_count=5,
        colorscale='Viridis',
        caps=dict(x_show=False, y_show=False, z_show=False)
    ))
    fig.update_layout(title="HOMO Orbital Isosurface")
    fig.show()



    # Find the index closest to z = 0
    x_index = np.argmin(np.abs(X[:,0,0]))

    # Extract XY slice of the orbital values at z = 0
    orbital_slice = orbital_vals_3d[x_index, :, :]

    # Create 2D plot using Plotly or matplotlib
    fig2 = go.Figure(data=go.Contour(
        z=orbital_slice,
        x=np.linspace(-4, 4, orbital_slice.shape[0]),  # x-axis
        y=np.linspace(-4, 4, orbital_slice.shape[1]),  # y-axis
        colorscale='RdBu',
        contours=dict(start=-0.1, end=0.1, size=0.01),
        colorbar=dict(title='Amplitude'),
    ))
    fig2.update_layout(
        title=f"2D Projection (XY slice at z = 0) for orbital {i}",
        xaxis_title="x (bohr)",
        yaxis_title="y (bohr)",
        width=600,
        height=500
    )
    fig2.show()



In [ ]:
import numpy as np

# Diagonal operator in original basis
eta = [1.0, 2.0, 3.0]

# Unitary transformation matrix (e.g., random orthogonal matrix)
U, _ = np.linalg.qr(np.random.randn(3, 3))

# Transform operator
eta_new = U.conj().T @ eta @ U  # shape (3,3)

print("Transformed operator matrix:")
print(np.round(eta_new, 3))


In [ ]:
import VariationalLangFirsov_QEDCI as vlfqedci
import numpy as np


mol_str  = """
        Li 0 0 0 
        H 0 0 1.4
        symmetry c1
        """


# Set computation options
psi4_options = {'basis': 'sto-3g',
                  'scf_type': 'pk',
                  'e_convergence': 1e-12}

lambda_vector = np.array([0.0,0.0,0.05])



qedhf  = vlfqedci.QED_HF(mol_str=mol_str, psi_4_options_dict=psi4_options)
qedhf.qed_hf(lambda_vector=lambda_vector)

wfn =  qedhf.wfn

# Get initial MO coefficients
C = wfn.Ca()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

omega = 1.0
d = 2.0
q = np.linspace(-3, 5, 500)

# Regular harmonic oscillator potential and energy
V = 0.5 * omega**2 * q**2
E0 = 0.5 * omega

# Regular ground state wavefunction squared (probability density)
psi_0 = ((omega / np.pi)**0.25 * np.exp(-0.5 * omega * q**2))**2

# Displaced harmonic oscillator potential and energy shift
V_d = 0.5 * omega**2 * (q - d)**2 - 0.5 * omega**2 * d**2
E_d = 0.5 * omega - 0.5 * omega**2 * d**2

# Displaced ground state wavefunction squared
psi_d = ((omega / np.pi)**0.25 * np.exp(-0.5 * omega * (q - d)**2))**2

# Scale wavefunctions so they are visible on the energy scale
scale = 2.5  # Controls wavefunction height

psi_0_scaled = psi_0 * scale + E0      # Shift wavefunction to correct energy
psi_d_scaled = psi_d * scale + E_d      # Shift displaced wf accordingly

plt.figure(figsize=(8,5))

# Plot potentials
plt.plot(q, V, 'k-', label='Regular Potential')
# plt.plot(q, V_d, 'k--', label='Displaced Potential')

# Fill under regular wavefunction
plt.fill_between(q, E0, psi_0_scaled, color='blue', alpha=0.3, label='Regular Ground State Fill')

# Fill under displaced wavefunction
# plt.fill_between(q, E_d, psi_d_scaled, color='cyan', alpha=0.3, label='Displaced Ground State Fill')

# Plot wavefunctions shifted to their energies
plt.plot(q, psi_0_scaled, 'b-', label='Regular Ground State')
# plt.plot(q, psi_d_scaled, 'b--', label='Displaced Ground State')

plt.xlabel('q (Photon coordinate)')
plt.ylabel('Energy')
plt.title('Harmonic Oscillator Potentials and Ground State Wavefunctions\n(Wavefunctions plotted at correct energies)')
plt.legend()
# Draw only x = 0 and y = 0 grid lines manually
plt.axhline(y=0, color='gray', linestyle='--', linewidth=1)
plt.axvline(x=0, color='gray', linestyle='--', linewidth=1)
plt.show()
